In [31]:
import dask
import dask.dataframe as dd
import pandas as pd
from  load_config import load_config
from dask_ml.wrappers import ParallelPostFit
from sklearn.pipeline import Pipeline as SKPipeline
from imblearn.over_sampling import SMOTE
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import cross_val_score,cross_validate
from imblearn.combine import SMOTEENN
from sklearn.preprocessing import PowerTransformer, OneHotEncoder, TargetEncoder
from dask.distributed import Client, LocalCluster
from dask_ml.model_selection import train_test_split
from sklearn.metrics import confusion_matrix,classification_report
from xgboost import XGBClassifier
from sklearn.impute import SimpleImputer

def read_excel(file_path):
   
   lazy_file = dask.delayed(pd.read_excel)(file_path)
   df = dd.from_delayed(lazy_file)
   
   return df

def clean_df(df):
  
  df['Total Charges']= df['Total Charges'].replace(' ', 0)
  df['Total Charges'] =df['Total Charges'].astype('float')
   
  df.columns = [col.strip().replace(' ', '_') for col in df.columns]
  df1= df.copy()
  remove_columns = ['Lat_Long','Latitude','Longitude','Total_Charges','Churn_Value','CLTV','Country','State','Streaming_Movies']
  df=df.drop(remove_columns,axis =1)		
  
  return df

def prepare_feature(df):
   df_num = df.select_dtypes(include ='number')
   df_cat = df.select_dtypes(exclude ='number').drop('Churn_Label',axis =1)
   all_features = [*df_cat.columns, *df_num.columns]
   X= df[all_features]
   y = df['Churn_Label'].map({'Yes':0,'No':1})
   
   return X,y, df_num, df_cat 
   
def build_preprocessor(df, num_col, cat_col):  
    
    # Numeric columns: fill with median
    transform_num = SKPipeline(steps=[
        ('impute_num', SimpleImputer(strategy='median')),
        ('scale_num', PowerTransformer(method='yeo-johnson'))
    ])
    
    # Categorical columns: fill with string 'missing'
    transform_cat = SKPipeline(steps=[
        ('impute_cat', SimpleImputer(strategy='constant', fill_value='missing')),
        ('scale_cat', TargetEncoder(random_state=42, cv=10))
    ])
    
    preprocess = ColumnTransformer(
        transformers=[
            ('nums', transform_num, num_col),
            ('cat', transform_cat, cat_col)
        ], 
        remainder='drop', 
        verbose=True
    )
    return preprocess

def build_model(preprocess):
    config = load_config("config.yaml")

    xgb_model = XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        n_estimators=1000,
        max_depth=3,
        learning_rate=0.01,
        subsample=1,
        colsample_bytree=1,
        n_jobs=-1,
        tree_method="hist",
        reg_alpha=0.01,
        reg_lambda=0.01,
    )

    model = SKPipeline(
        steps=[
            ("preprocess", preprocess),
            ("imbalance", SMOTE(random_state=42)),
            ("xgb", xgb_model),
        ]
    )
    return model

from imblearn.over_sampling import SMOTE

import numpy as np

def main():

    config = load_config('config.yaml')
    file_path = config['data']['path']
    
    df = read_excel(file_path)
    df = clean_df(df)
    
    # Convert Dask dataframe to pandas
    df = df.compute()
    
    # Fill all NA values before processing
    df = df.fillna('missing')
    
    X, y, df_num, df_cat = prepare_feature(df)

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30,
                                                         random_state=42, shuffle=True)

    preprocessor = build_preprocessor(df, df_num.columns.to_list(), df_cat.columns.to_list())

    # Apply SMOTE to training data BEFORE fitting
    X_train_processed = preprocessor.fit_transform(X_train, y_train)
    
    # Convert sparse matrix to dense array and ensure float dtype
    if hasattr(X_train_processed, 'toarray'):  # If sparse matrix
        X_train_processed = X_train_processed.toarray()
    X_train_processed = np.asarray(X_train_processed, dtype='float64')
    
    smote = SMOTE(random_state=42)
    X_train_resampled, y_train_resampled = smote.fit_resample(X_train_processed, y_train)

    # Now build and fit just the XGBClassifier (no SMOTE in pipeline)
    config = load_config("config.yaml")
    xgb_model = XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        n_estimators=1000,
        max_depth=3,
        learning_rate=0.01,
        subsample=1,
        colsample_bytree=1,
        n_jobs=-1,
        tree_method="hist",
        reg_alpha=0.01,
        reg_lambda=0.01,
    )
    
    xgb_model.fit(X_train_resampled, y_train_resampled)

    # Preprocess test data and predict
    X_test_processed = preprocessor.transform(X_test)
    
    # Convert sparse matrix to dense array and ensure float dtype
    if hasattr(X_test_processed, 'toarray'):  # If sparse matrix
        X_test_processed = X_test_processed.toarray()
    X_test_processed = np.asarray(X_test_processed, dtype='float64')
    
    y_pred = xgb_model.predict(X_test_processed)
    y_pred_prob = xgb_model.predict_proba(X_test_processed)[:, 1]

    print(confusion_matrix(y_test, y_pred))
    print(xgb_model.score(X_test,y_test))


if __name__ == "__main__":
    main()


[ColumnTransformer] .......... (1 of 2) Processing nums, total=   0.0s
[ColumnTransformer] ........... (2 of 2) Processing cat, total=   0.1s


d:\DATASCIENCE\PRACTICAL\Lecture\Level4\telco Churn\.venv\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:341: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


[[ 588    0]
 [   0 1525]]


ValueError: DataFrame.dtypes for data must be int, float, bool or category. When categorical type is supplied, the experimental DMatrix parameter`enable_categorical` must be set to `True`.  Invalid columns:CustomerID: string, City: string, Gender: string, Senior_Citizen: string, Partner: string, Dependents: string, Phone_Service: string, Multiple_Lines: string, Internet_Service: string, Online_Security: string, Online_Backup: string, Device_Protection: string, Tech_Support: string, Streaming_TV: string, Contract: string, Paperless_Billing: string, Payment_Method: string, Churn_Reason: string

In [ ]:
import dask
import dask.dataframe as dd
import pandas as pd
import numpy as np
from  load_config import load_config
from dask_ml.wrappers import ParallelPostFit
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import cross_val_score,cross_validate,StratifiedKFold
from imblearn.combine import SMOTEENN
from sklearn.preprocessing import PowerTransformer, OneHotEncoder, TargetEncoder
from dask.distributed import Client, LocalCluster
from dask_ml.model_selection import train_test_split
from sklearn.metrics import confusion_matrix,classification_report
from xgboost import XGBClassifier
import joblib
import warnings
warnings.filterwarnings('ignore')

config = load_config('config.yaml')
remove_col = config['data']['remove_columns']			
file_path = config['data']['path']
random = config['data']['random_state']
test_size = config['data']['test_size']
job = config['data']['n_jobs']
n_splits = config['data']['n_splits']
config_objective = config['data']['objective']
config_eval_metric = config['data']['eval_metric']
config_n_estimators = config['data']['n_estimators']
config_max_depth = config['data']['max_depth']
config_learning_rate = config['data']['learning_rate']
config_subsample = config['data']['subsample']
config_colsample_bytree = config['data']['colsample_bytree']
config_n_jobs = config['data']['n_jobs']
config_tree_method = config['data']['tree_method']
config_reg_alpha = config['data']['reg_alpha']
config_reg_lambda = config['data']['reg_lambda']
config_random_state = config['data']['random_state']
model_path=config['data']['models']


def dask_cluster():
    # Create a local cluster with multiple workers
    cluster = LocalCluster(
        n_workers=4,              # number of worker processes
        threads_per_worker=2,     # threads per worker
        memory_limit="4GB",       # memory cap per worker
        dashboard_address=":8787" # web UI for monitoring
    )
    client = Client(cluster)
    print(client)  # shows cluster info
    return client


def read_excel(file_path):
   
   lazy_file = dask.delayed(pd.read_excel)(file_path)
   df = dd.from_delayed(lazy_file)
   
   return df

def clean_df(df):
  
  df['Total Charges']= df['Total Charges'].replace(' ', 0)
  df['Total Charges'] =df['Total Charges'].astype('float')
   
  df.columns = [col.strip().replace(' ', '_') for col in df.columns]
  df1= df.copy()
  remove_columns= remove_col
  df=df.drop(remove_columns,axis =1)		
  
  return df

def prepare_feature(df):
   df_num = df.select_dtypes(include ='number')
   df_cat = df.select_dtypes(exclude ='number').drop('Churn_Label',axis =1)
   all_features = [*df_cat.columns, *df_num.columns]
   X= df[all_features]
   y = df['Churn_Label'].map({'Yes':0,'No':1})
   
   return X,y, df_num, df_cat 
   
def build_preprocessor(df,num_col, cat_col):  
   
   transform_num = Pipeline(steps=[('scale_num',PowerTransformer(method='yeo-johnson'))])
   transform_cat = Pipeline(steps=[('scale_cat',TargetEncoder(random_state=config['data']['random_state'], cv=config['data']['n_splits']))])
   
   preprocess = ColumnTransformer(transformers=[('nums', transform_num,num_col),
																																																('cat',transform_cat,cat_col)
																																																], remainder='drop', verbose=True)
   return preprocess

def build_model(preprocess):   
    

    xgb_model = XGBClassifier(
        objective=config_objective,
        eval_metric=config_eval_metric,
        n_estimators=config_n_estimators,
        max_depth=config_max_depth,
        learning_rate=config_learning_rate,
        subsample=config_subsample,
        colsample_bytree=config_colsample_bytree,
        n_jobs=config_n_jobs,
        tree_method=config_tree_method,
        reg_alpha=config_reg_alpha,
        reg_lambda=config_reg_lambda
    )

    model = Pipeline(
        steps=[
            ("preprocess", preprocess),
            ("imbalance", SMOTE(random_state=config_random_state)),
            ("xgb", xgb_model),
        ]
    )
    return model

def create_pkl_file(model):
    joblib.dump(model, model_path)
    print(f"Model saved successfully at {model_path}")

def main():   
   
   client =dask_cluster()
   
   df = read_excel(file_path)

   df= clean_df(df)
   
   X, y, df_num,df_cat = prepare_feature(df)

   X= X.persist()
   y= y.persist()

   X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=test_size,
                                                    random_state =random, shuffle =True)

   preprocessor = build_preprocessor(df,df_num.columns.to_list(),df_cat.columns.to_list())

   model = build_model(preprocessor)

   model.fit(X_train.compute(),y_train.compute())

   y_pred = model.predict(X_test.compute())

   y_pred_prob = model.predict_proba(X_test.compute())[:,1]
   
   print(confusion_matrix(y_test,y_pred))
   print(classification_report(y_test,y_pred))
   cv_str=StratifiedKFold(n_splits=n_splits,shuffle = True,random_state=random)
   score_train = cross_val_score(model,X=X_train.compute(),y=y_train.compute(),cv = cv_str,n_jobs=job,error_score= 'raise',scoring='roc_auc')
   print(np.mean(score_train))
   score_test = cross_val_score(model,X=X_test.compute(),y=y_test.compute(),cv = cv_str,n_jobs=job,error_score= 'raise',scoring='roc_auc')
   print(np.mean(score_test))
   create_pkl_file(model)
   client.close()
			

if __name__ == "__main__":
   main()

<Client: 'tcp://127.0.0.1:52693' processes=4 threads=8, memory=14.90 GiB>
[ColumnTransformer] .......... (1 of 2) Processing nums, total=   0.1s
[ColumnTransformer] ........... (2 of 2) Processing cat, total=   0.1s
[[ 519   15]
 [ 152 1383]]
              precision    recall  f1-score   support

           0       0.77      0.97      0.86       534
           1       0.99      0.90      0.94      1535

    accuracy                           0.92      2069
   macro avg       0.88      0.94      0.90      2069
weighted avg       0.93      0.92      0.92      2069

0.9818042851752466
0.9856294299690527
Model saved successfully at ../model/Telco_customer_churn.pkl


2026-08-21 10:43:21,056 - tornado.application - ERROR - Uncaught exception GET /status/ws (127.0.0.1)
HTTPServerRequest(protocol='http', host='localhost:8787', method='GET', uri='/status/ws', version='HTTP/1.1', remote_ip='127.0.0.1')
Traceback (most recent call last):
  File "d:\DATASCIENCE\PRACTICAL\Lecture\Level4\telco Churn\.venv\Lib\site-packages\tornado\websocket.py", line 965, in _accept_connection
    open_result = handler.open(*handler.open_args, **handler.open_kwargs)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\DATASCIENCE\PRACTICAL\Lecture\Level4\telco Churn\.venv\Lib\site-packages\tornado\web.py", line 3415, in wrapper
    return method(self, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\DATASCIENCE\PRACTICAL\Lecture\Level4\telco Churn\.venv\Lib\site-packages\bokeh\server\views\ws.py", line 157, in open
    raise ProtocolError("Token is expired. Configure the app with a larger value for --session-token-expirati

NameError: name 'X' is not defined